In [7]:
import numpy as np
import pandas as pd
import random

# Load the dataset
file_path = 'data/dg14bus.xlsx'
data = pd.read_excel(file_path, sheet_name='Sheet1')

# Extract the relevant columns: 'from', 'to', 'resistansi', 'reaktansi', 'suseptansi', 'MVA'
line_data = data[['from', 'to', 'resistansi', 'reaktansi', 'suseptansi', 'MVA']].values

# Number of buses and lines
n_lines = line_data.shape[0]
buses = np.unique(np.concatenate([line_data[:, 0], line_data[:, 1]]))
n_buses = len(buses)

# Maximum DG capacity in kW and converted to MW
max_dg_capacity_kw = 500  # Maximum DG capacity in kW
max_dg_capacity_mw = max_dg_capacity_kw / 1e3  # Convert kW to MW

# Define the bounds for DG placement (limits of DG capacity)
lower_bound = np.zeros(n_buses)
upper_bound = np.array([max_dg_capacity_mw] * n_buses)  # Set max DG capacity per bus to 500 kW (0.5 MW)

# Objective function: Minimize power loss based on DG placement, accounting for MVA limits and susceptance
def calculate_line_loss(P_from, P_to, resistance, reactance, susceptance, MVA):
    # Impedance with susceptance included
    impedance = np.sqrt(resistance**2 + (reactance - susceptance)**2)
    # Simplified power flow model, assume V = 1.0 (per-unit)
    current_flow = (P_from - P_to) / impedance
    # Power loss calculation
    power_loss = current_flow**2 * resistance
    
    # Ensure the power flow does not exceed the line's MVA rating
    apparent_power_flow = abs(current_flow)  # Magnitude of power flow
    if apparent_power_flow > MVA:
        # If power flow exceeds MVA, cap the power loss
        apparent_power_flow = MVA
        power_loss = MVA**2 * resistance / impedance**2

    return power_loss

# Calculate total losses across all buses
def calculate_total_losses(dg_placement):
    total_loss = 0
    bus_generation = {bus: dg_placement[i] for i, bus in enumerate(buses)}
    
    for line in line_data:
        from_bus = int(line[0])
        to_bus = int(line[1])
        resistance = line[2]
        reactance = line[3]
        susceptance = line[4]
        MVA = line[5]
        power_from = bus_generation.get(from_bus, 0)
        power_to = bus_generation.get(to_bus, 0)
        total_loss += calculate_line_loss(power_from, power_to, resistance, reactance, susceptance, MVA)
    
    return total_loss

# Calculate losses considering only the top 3 DG placements
def calculate_total_losses_top_3(dg_placement, top_3_indices):
    total_loss = 0
    bus_generation = {bus: dg_placement[i] if i in top_3_indices else 0 for i, bus in enumerate(buses)}
    
    for line in line_data:
        from_bus = int(line[0])
        to_bus = int(line[1])
        resistance = line[2]
        reactance = line[3]
        susceptance = line[4]
        MVA = line[5]
        power_from = bus_generation.get(from_bus, 0)
        power_to = bus_generation.get(to_bus, 0)
        total_loss += calculate_line_loss(power_from, power_to, resistance, reactance, susceptance, MVA)
    
    return total_loss

# Grey Wolf Optimizer (GWO) implementation
class GWO:
    def __init__(self, objective_function, lb, ub, n_buses, max_iter):
        self.obj_function = objective_function
        self.lb = lb  # lower bounds
        self.ub = ub  # upper bounds
        self.n_buses = n_buses
        self.max_iter = max_iter

    def optimize(self):
        # Initialize wolf positions
        positions = np.random.uniform(self.lb, self.ub, (self.n_buses, n_buses))
        alpha_wolf, beta_wolf, delta_wolf = np.zeros(n_buses), np.zeros(n_buses), np.zeros(n_buses)
        alpha_score, beta_score, delta_score = float("inf"), float("inf"), float("inf")

        # Main loop
        for t in range(self.max_iter):
            for i in range(self.n_buses):
                # Calculate the fitness of each wolf
                fitness = self.obj_function(positions[i])

                # Update alpha, beta, and delta wolves
                if fitness < alpha_score:
                    alpha_score, beta_score, delta_score = fitness, alpha_score, beta_score
                    alpha_wolf, beta_wolf, delta_wolf = positions[i], alpha_wolf, beta_wolf
                elif fitness < beta_score:
                    beta_score, delta_score = fitness, beta_score
                    beta_wolf, delta_wolf = positions[i], beta_wolf
                elif fitness < delta_score:
                    delta_score = fitness
                    delta_wolf = positions[i]

            # Update positions of the wolves
            a = 2 - t * (2 / self.max_iter)  # Decrease linearly from 2 to 0

            for i in range(self.n_buses):
                for j in range(n_buses):
                    # Position update based on alpha, beta, and delta wolves
                    r1, r2 = random.random(), random.random()
                    A1, C1 = 2 * a * r1 - a, 2 * r2
                    D_alpha = abs(C1 * alpha_wolf[j] - positions[i][j])
                    X1 = alpha_wolf[j] - A1 * D_alpha

                    r1, r2 = random.random(), random.random()
                    A2, C2 = 2 * a * r1 - a, 2 * r2
                    D_beta = abs(C2 * beta_wolf[j] - positions[i][j])
                    X2 = beta_wolf[j] - A2 * D_beta

                    r1, r2 = random.random(), random.random()
                    A3, C3 = 2 * a * r1 - a, 2 * r2
                    D_delta = abs(C3 * delta_wolf[j] - positions[i][j])
                    X3 = delta_wolf[j] - A3 * D_delta

                    # Update position based on average
                    positions[i][j] = (X1 + X2 + X3) / 3

                    # Ensure wolves stay within bounds
                    positions[i][j] = np.clip(positions[i][j], self.lb[j], self.ub[j])

        return alpha_wolf, alpha_score

# Step 1: Calculate pre-optimization losses (without any DG)
pre_optimization_losses_mw = calculate_total_losses(np.zeros(n_buses)) / 1e3  # Convert to MW

# Step 2: Initialize the GWO optimizer
gwo = GWO(calculate_total_losses, lower_bound, upper_bound, n_buses, max_iter=100)

# Step 3: Run the optimization
best_dg_position, post_optimization_loss = gwo.optimize()

# Step 4: Convert optimal DG placement to kW
optimal_dg_capacity_kW = np.sum(best_dg_position) * 1e3  # Convert MW to kW

# Step 5: Output the results in kW
post_optimization_losses_mw = post_optimization_loss / 1e3  # Convert to MW

# Find the indices of the three highest DG placements after optimization
top_3_dg_indices = np.argsort(-best_dg_position)[:3]  # Sort and take top 3 indices
top_3_dg_buses = buses[top_3_dg_indices]  # Buses corresponding to top 3 DG placements
top_3_dg_capacities = best_dg_position[top_3_dg_indices] * 1e3  # Convert MW to kW
post_optimization_loss_top_3 = calculate_total_losses_top_3(best_dg_position, top_3_dg_indices) / 1e3  # Convert to MW

# Print results
print(f"Pre-optimization losses (MW): {pre_optimization_losses_mw}")
print("Optimal DG placements (Top 3):")
for i, (bus, capacity) in enumerate(zip(top_3_dg_buses, top_3_dg_capacities), start=1):
    print(f"{i}. Bus {int(bus)} - DG Capacity: {capacity:.2f} kW")
print(f"Post-optimization losses (considering only Top 3 DG placements, MW): {post_optimization_loss_top_3}")
print(f"Post-optimization losses (all placements, MW): {post_optimization_losses_mw}")

Pre-optimization losses (MW): 0.0
Optimal DG placements (Top 3):
1. Bus 13 - DG Capacity: 36.55 kW
2. Bus 5 - DG Capacity: 24.94 kW
3. Bus 8 - DG Capacity: 17.30 kW
Post-optimization losses (considering only Top 3 DG placements, MW): 1.674174644640029e-05
Post-optimization losses (all placements, MW): 2.434882284687648e-06
